# Experiment Metrics Comparison

This notebook loads `metrics.json` files from multiple thesis experiments, extracts train/test/validation metrics, creates comparison tables, and saves report-ready figures/tables.

Update `EXPERIMENT_METRICS` if you add more experiment folders.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis"
)

MODELS_DIR = PROJECT_ROOT / "models" / "trained"

metadata_files = sorted(MODELS_DIR.glob("*/metadata.json"))

print(f"Found {len(metadata_files)} metadata files.\n")

rows = []

for metadata_path in metadata_files:

    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    perf = metadata.get("performance", {})
    training = metadata.get("training", {})
    preprocessing = metadata.get("preprocessing", {})

    rows.append(
        {
            "Model ID": metadata.get("model_id"),
            "Display Name": metadata.get("display_name"),
            "Framework": metadata.get("framework"),
            "Model Type": metadata.get("model_type"),
            "Architecture": metadata.get("architecture", ""),

            "Validation Accuracy":
                perf.get("validation_accuracy", perf.get("val_accuracy")),

            "Validation Balanced Accuracy":
                perf.get(
                    "validation_balanced_accuracy",
                    perf.get("val_balanced_accuracy"),
                ),

            "Validation Macro F1":
                perf.get(
                    "validation_macro_f1",
                    perf.get("val_macro_f1"),
                ),

            "Test Accuracy":
                perf.get("test_accuracy"),

            "Test Balanced Accuracy":
                perf.get("test_balanced_accuracy"),

            "Test Macro F1":
                perf.get("test_macro_f1"),

            "Best Epoch":
                perf.get("best_epoch"),

            "Batch Size":
                training.get("batch_size"),

            "Learning Rate":
                training.get("learning_rate"),

            "Fluorescence Threshold":
                preprocessing.get("fluorescence_threshold"),

            "Metadata File":
                metadata_path,

            "Model Folder":
                metadata_path.parent.name,
        }
    )

metrics_df = (
    pd.DataFrame(rows)
    .sort_values(
        by="Test Balanced Accuracy",
        ascending=False,
        na_position="last",
    )
    .reset_index(drop=True)
)

metrics_df.insert(
    0,
    "Rank",
    metrics_df["Test Balanced Accuracy"]
        .rank(method="dense", ascending=False)
        .astype("Int64"),
)

display(metrics_df)

print(f"\nLoaded metrics for {len(metrics_df)} trained models.")

Found 7 metadata files.



,Rank,Model ID,Display Name,Framework,Model Type,Architecture,Validation Accuracy,Validation Balanced Accuracy,Validation Macro F1,Test Accuracy,Test Balanced Accuracy,Test Macro F1,Best Epoch,Batch Size,Learning Rate,Fluorescence Threshold,Metadata File,Model Folder
0,1,exp05_multimodal_species_v1,Exp05 Multimodal CNN,NaN,torch_multimodal,MultimodalDeepClassifier,NaN,NaN,NaN,0.876020,0.768510,0.768985,20.0,NaN,NaN,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,exp05_multimodal_species_v1
1,2,exp03_tuned_rf_v1,Exp03 Tuned RF,sklearn,sklearn_random_forest,,NaN,NaN,NaN,0.853181,0.715916,0.721003,NaN,NaN,NaN,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,exp03_tuned_rf_v1
2,3,exp02_grouped_engineered_rf_v1,Exp02 Grouped Engineered RF,sklearn,sklearn_random_forest,,NaN,NaN,NaN,0.833605,0.664343,0.653866,NaN,NaN,NaN,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,exp02_grouped_engineered_rf_v1
3,4,exp01_grouped_paper_rf_v1,Experiment 01 Grouped Paper RF,sklearn,sklearn_random_forest,,NaN,0.685260,NaN,NaN,0.659046,0.644171,NaN,NaN,NaN,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,exp01_grouped_paper_rf_v1
4,5,exp04_baseline_cnn_v1,Experiment 04 Baseline CNN,torch,torch_cnn1d,BaselineCNN1D,NaN,0.451098,0.444885,NaN,0.443002,0.440949,19.0,128.0,0.001,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,exp04_baseline_cnn_v1
5,<NA>,exp00_paper_rf_v1,Exp00 Paper RF Baseline,sklearn,sklearn_random_forest,,0.806588,0.654017,0.640995,NaN,NaN,NaN,NaN,NaN,NaN,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,exp00_paper_rf_v1
6,<NA>,exp06_robust_multimodal_species_v1,Exp06 Robust Multimodal CNN,NaN,torch_multimodal_robust,RobustMultimodalClassifier,NaN,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,exp06_robust_multimodal_species_v1



Loaded metrics for 7 trained models.


## Load metrics JSON files

In [2]:
from pathlib import Path
from typing import Any
import json

import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis"
)

MODELS_DIR = PROJECT_ROOT / "models" / "trained"

metadata_files = sorted(MODELS_DIR.glob("*/metadata.json"))

def load_json(path: Path) -> dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def safe_get(d: dict[str, Any], *keys, default=None):
    current = d
    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


rows = []
missing_files = []

for metadata_path in metadata_files:
    try:
        metadata = load_json(metadata_path)

        rows.append({
            "model_folder": metadata_path.parent.name,
            "model_id": metadata.get("model_id"),
            "display_name": metadata.get("display_name"),
            "framework": metadata.get("framework"),
            "model_type": metadata.get("model_type"),
            "architecture": metadata.get("architecture"),

            "validation_accuracy": safe_get(metadata, "performance", "validation_accuracy"),
            "validation_balanced_accuracy": safe_get(metadata, "performance", "validation_balanced_accuracy"),
            "validation_macro_f1": safe_get(metadata, "performance", "validation_macro_f1"),

            "val_accuracy": safe_get(metadata, "performance", "val_accuracy"),
            "val_balanced_accuracy": safe_get(metadata, "performance", "val_balanced_accuracy"),
            "val_macro_f1": safe_get(metadata, "performance", "val_macro_f1"),

            "test_accuracy": safe_get(metadata, "performance", "test_accuracy"),
            "test_balanced_accuracy": safe_get(metadata, "performance", "test_balanced_accuracy"),
            "test_macro_f1": safe_get(metadata, "performance", "test_macro_f1"),

            "best_epoch": safe_get(metadata, "performance", "best_epoch"),
            "fluorescence_threshold": safe_get(metadata, "preprocessing", "fluorescence_threshold"),

            "metadata_path": str(metadata_path),
        })

    except Exception as exc:
        missing_files.append({
            "metadata_path": str(metadata_path),
            "error": repr(exc),
        })


metrics_df = pd.DataFrame(rows)

metrics_df["final_validation_accuracy"] = metrics_df["validation_accuracy"].fillna(
    metrics_df["val_accuracy"]
)
metrics_df["final_validation_balanced_accuracy"] = metrics_df[
    "validation_balanced_accuracy"
].fillna(metrics_df["val_balanced_accuracy"])
metrics_df["final_validation_macro_f1"] = metrics_df["validation_macro_f1"].fillna(
    metrics_df["val_macro_f1"]
)

metrics_df = metrics_df.sort_values(
    "test_balanced_accuracy",
    ascending=False,
    na_position="last",
).reset_index(drop=True)

display(metrics_df)

print(f"Loaded {len(metrics_df)} model metadata files")
if missing_files:
    display(pd.DataFrame(missing_files))

,model_folder,model_id,display_name,framework,model_type,architecture,validation_accuracy,validation_balanced_accuracy,validation_macro_f1,val_accuracy,...,val_macro_f1,test_accuracy,test_balanced_accuracy,test_macro_f1,best_epoch,fluorescence_threshold,metadata_path,final_validation_accuracy,final_validation_balanced_accuracy,final_validation_macro_f1
0,exp05_multimodal_species_v1,exp05_multimodal_species_v1,Exp05 Multimodal CNN,NaN,torch_multimodal,MultimodalDeepClassifier,NaN,NaN,NaN,None,...,NaN,0.876020,0.768510,0.768985,20.0,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,None,NaN,NaN
1,exp03_tuned_rf_v1,exp03_tuned_rf_v1,Exp03 Tuned RF,sklearn,sklearn_random_forest,NaN,NaN,NaN,NaN,None,...,NaN,0.853181,0.715916,0.721003,NaN,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,None,NaN,NaN
2,exp02_grouped_engineered_rf_v1,exp02_grouped_engineered_rf_v1,Exp02 Grouped Engineered RF,sklearn,sklearn_random_forest,NaN,NaN,NaN,NaN,None,...,NaN,0.833605,0.664343,0.653866,NaN,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,None,NaN,NaN
3,exp01_grouped_paper_rf_v1,exp01_grouped_paper_rf_v1,Experiment 01 Grouped Paper RF,sklearn,sklearn_random_forest,NaN,NaN,NaN,NaN,None,...,NaN,NaN,0.659046,0.644171,NaN,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,None,0.685260,NaN
4,exp04_baseline_cnn_v1,exp04_baseline_cnn_v1,Experiment 04 Baseline CNN,torch,torch_cnn1d,BaselineCNN1D,NaN,NaN,NaN,None,...,0.444885,NaN,0.443002,0.440949,19.0,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,None,0.451098,0.444885
5,exp00_paper_rf_v1,exp00_paper_rf_v1,Exp00 Paper RF Baseline,sklearn,sklearn_random_forest,NaN,0.806588,0.654017,0.640995,None,...,NaN,NaN,NaN,NaN,NaN,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,0.806588,0.654017,0.640995
6,exp06_robust_multimodal_species_v1,exp06_robust_multimodal_species_v1,Exp06 Robust Multimodal CNN,NaN,torch_multimodal_robust,RobustMultimodalClassifier,NaN,NaN,NaN,None,...,NaN,NaN,NaN,NaN,30.0,2000.0,C:\Users\chris\OneDrive\Documents\Universitat ...,None,NaN,NaN


Loaded 7 model metadata files


## Flatten experiment metrics

The metrics JSON structure can vary slightly by experiment. This cell searches for common split names such as `train`, `test`, `test_tuning`, `validation`, and `validation_final`.


In [3]:
POSSIBLE_SPLITS = [
    "train",
    "test",
    "test_tuning",
    "validation",
    "validation_final",
    "val",
    "holdout",
]

METRIC_KEYS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "precision",
    "recall",
]


def infer_model_label(experiment_id: str, metrics: dict[str, Any]) -> str:
    model_id = safe_get(metrics, "experiment", "model_id")
    model_type = safe_get(metrics, "experiment", "model_type")
    display_name = safe_get(metrics, "experiment", "name")

    if model_id:
        return str(model_id)
    if display_name:
        return str(display_name)
    if model_type:
        return f"{experiment_id} ({model_type})"
    return experiment_id


def extract_split_rows(experiment_id: str, metrics: dict[str, Any]) -> list[dict[str, Any]]:
    rows = []
    model_label = infer_model_label(experiment_id, metrics)

    for split_name in POSSIBLE_SPLITS:
        split_metrics = metrics.get(split_name)

        if not isinstance(split_metrics, dict):
            continue

        row = {
            "experiment_id": experiment_id,
            "model_label": model_label,
            "split": split_name,
        }

        for key in METRIC_KEYS:
            row[key] = split_metrics.get(key)

        best_config = metrics.get("best_config", {})
        if isinstance(best_config, dict):
            row["feature_set"] = best_config.get("feature_set")
            row["n_estimators"] = best_config.get("n_estimators")
            row["max_depth"] = best_config.get("max_depth")
            row["n_features"] = best_config.get("n_features")

        exp_info = metrics.get("experiment", {})
        if isinstance(exp_info, dict):
            row["model_type"] = exp_info.get("model_type")
            row["split_protocol"] = exp_info.get("split_protocol")
            row["notes"] = exp_info.get("note") or exp_info.get("notes")

        rows.append(row)

    # Fallback for flat metrics files
    if not rows:
        row = {
            "experiment_id": experiment_id,
            "model_label": model_label,
            "split": "reported",
        }
        for key in METRIC_KEYS:
            row[key] = metrics.get(key)
        rows.append(row)

    return rows


rows = []
for experiment_id, metrics in loaded_metrics.items():
    rows.extend(extract_split_rows(experiment_id, metrics))

metrics_df = pd.DataFrame(rows)

for col in METRIC_KEYS:
    if col in metrics_df.columns:
        metrics_df[col] = pd.to_numeric(metrics_df[col], errors="coerce")

metrics_df

NameError: name 'loaded_metrics' is not defined

## Save full metrics table

In [ ]:
# ------------------------------------------------------------------
# Save complete model performance table
# ------------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "model_comparison"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_file = OUTPUT_DIR / "all_model_metrics.csv"
latex_file = OUTPUT_DIR / "all_model_metrics.tex"
markdown_file = OUTPUT_DIR / "all_model_metrics.md"
json_file = OUTPUT_DIR / "all_model_metrics.json"

# Save CSV
metrics_df.to_csv(csv_file, index=False)

# Save JSON
metrics_df.to_json(
    json_file,
    orient="records",
    indent=4,
)

# Save Markdown
metrics_df.to_markdown(
    markdown_file,
    index=False,
)

# Save LaTeX
metrics_df.to_latex(
    latex_file,
    index=False,
    float_format="%.4f",
    na_rep="-",
    caption="Summary of model performance across all experiments.",
    label="tab:model_comparison",
)

print("Saved:")
print(f"  CSV      : {csv_file}")
print(f"  JSON     : {json_file}")
print(f"  Markdown : {markdown_file}")
print(f"  LaTeX    : {latex_file}")

ImportError: `Import tabulate` failed.  Use pip or conda to install the tabulate package.

## Report-ready summary table

This table prioritizes the final validation split when present. If not available, it falls back to test/test_tuning/reported metrics.


In [ ]:
SPLIT_PRIORITY = {
    "validation_final": 0,
    "validation": 1,
    "val": 2,
    "test": 3,
    "test_tuning": 4,
    "holdout": 5,
    "reported": 6,
    "train": 7,
}

summary_df = metrics_df.copy()
summary_df["split_priority"] = summary_df["split"].map(SPLIT_PRIORITY).fillna(99)

summary_df = (
    summary_df
    .sort_values(["experiment_id", "split_priority"])
    .groupby("experiment_id", as_index=False)
    .first()
)

summary_cols = [
    "experiment_id",
    "model_label",
    "split",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "feature_set",
    "n_features",
    "model_type",
    "split_protocol",
]

summary_cols = [c for c in summary_cols if c in summary_df.columns]
summary_df = summary_df[summary_cols].sort_values("balanced_accuracy", ascending=False, na_position="last")

summary_df.to_csv(OUTPUT_DIR / "experiment_metrics_summary_report_table.csv", index=False)

with pd.ExcelWriter(metrics_xlsx, mode="a", if_sheet_exists="replace") as writer:
    summary_df.to_excel(writer, sheet_name="summary_report", index=False)

summary_df

## Wide comparison table

In [ ]:
wide_df = metrics_df.pivot_table(
    index=["experiment_id", "model_label"],
    columns="split",
    values=["accuracy", "balanced_accuracy", "macro_f1"],
    aggfunc="first",
)

wide_df.columns = [f"{metric}_{split}" for metric, split in wide_df.columns]
wide_df = wide_df.reset_index()

wide_df.to_csv(OUTPUT_DIR / "experiment_metrics_wide_comparison.csv", index=False)

with pd.ExcelWriter(metrics_xlsx, mode="a", if_sheet_exists="replace") as writer:
    wide_df.to_excel(writer, sheet_name="wide_comparison", index=False)

wide_df

## Figure 1: Model comparison by balanced accuracy

In [ ]:
plot_df = summary_df.dropna(subset=["balanced_accuracy"]).copy()
plot_df = plot_df.sort_values("balanced_accuracy", ascending=True)

plt.figure(figsize=(10, max(4, 0.45 * len(plot_df))))
plt.barh(plot_df["experiment_id"], plot_df["balanced_accuracy"])
plt.xlabel("Balanced accuracy")
plt.ylabel("Experiment")
plt.title("Model performance comparison")
plt.xlim(0, 1)
plt.tight_layout()

fig_path = OUTPUT_DIR / "figure_model_balanced_accuracy_comparison.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)

## Figure 2: Accuracy, balanced accuracy, and macro F1

In [ ]:
plot_metrics = ["accuracy", "balanced_accuracy", "macro_f1"]
plot_long = summary_df.melt(
    id_vars=["experiment_id", "split"],
    value_vars=[m for m in plot_metrics if m in summary_df.columns],
    var_name="metric",
    value_name="score",
).dropna(subset=["score"])

pivot_for_plot = plot_long.pivot(index="experiment_id", columns="metric", values="score")
pivot_for_plot = pivot_for_plot.sort_values("balanced_accuracy", ascending=True)

ax = pivot_for_plot.plot(kind="barh", figsize=(11, max(4, 0.55 * len(pivot_for_plot))))
ax.set_xlabel("Score")
ax.set_ylabel("Experiment")
ax.set_title("Model comparison across main classification metrics")
ax.set_xlim(0, 1)
plt.tight_layout()

fig_path = OUTPUT_DIR / "figure_model_metric_grouped_bar.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)

## Figure 3: Train vs evaluation gap

This plot is useful for showing overfitting. It compares train balanced accuracy against the final/report split when both are available.


In [ ]:
train_df = metrics_df[metrics_df["split"] == "train"][
    ["experiment_id", "balanced_accuracy", "accuracy", "macro_f1"]
].rename(columns={
    "balanced_accuracy": "train_balanced_accuracy",
    "accuracy": "train_accuracy",
    "macro_f1": "train_macro_f1",
})

eval_df = summary_df[
    ["experiment_id", "split", "balanced_accuracy", "accuracy", "macro_f1"]
].rename(columns={
    "split": "eval_split",
    "balanced_accuracy": "eval_balanced_accuracy",
    "accuracy": "eval_accuracy",
    "macro_f1": "eval_macro_f1",
})

gap_df = train_df.merge(eval_df, on="experiment_id", how="inner")
gap_df["balanced_accuracy_gap"] = gap_df["train_balanced_accuracy"] - gap_df["eval_balanced_accuracy"]

gap_df.to_csv(OUTPUT_DIR / "train_vs_eval_gap.csv", index=False)

if not gap_df.empty:
    gap_plot = gap_df.sort_values("balanced_accuracy_gap", ascending=True)

    plt.figure(figsize=(10, max(4, 0.45 * len(gap_plot))))
    plt.barh(gap_plot["experiment_id"], gap_plot["balanced_accuracy_gap"])
    plt.xlabel("Train balanced accuracy - evaluation balanced accuracy")
    plt.ylabel("Experiment")
    plt.title("Estimated overfitting gap by experiment")
    plt.tight_layout()

    fig_path = OUTPUT_DIR / "figure_train_eval_balanced_accuracy_gap.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)

gap_df

## Optional: confusion matrices from metrics JSON

If the metrics files contain confusion matrices, this cell saves one figure per experiment/split.


In [ ]:
def get_confusion_matrix(metrics: dict[str, Any], split: str):
    split_metrics = metrics.get(split)
    if isinstance(split_metrics, dict):
        cm = split_metrics.get("confusion_matrix")
        if cm is not None:
            return np.asarray(cm)
    return None


def get_class_names(metrics: dict[str, Any]):
    class_names = safe_get(metrics, "experiment", "class_names")
    if class_names:
        return class_names

    # Try classification report keys
    for split in POSSIBLE_SPLITS:
        report = safe_get(metrics, split, "classification_report")
        if isinstance(report, dict):
            names = [
                k for k in report.keys()
                if k not in {"accuracy", "macro avg", "weighted avg"}
            ]
            if names:
                return names

    return None


confusion_output_dir = OUTPUT_DIR / "confusion_matrices"
confusion_output_dir.mkdir(parents=True, exist_ok=True)

saved_confusion_figures = []

for experiment_id, metrics in loaded_metrics.items():
    class_names = get_class_names(metrics)

    for split in POSSIBLE_SPLITS:
        cm = get_confusion_matrix(metrics, split)
        if cm is None:
            continue

        plt.figure(figsize=(7, 6))
        plt.imshow(cm)
        plt.title(f"{experiment_id} — {split} confusion matrix")
        plt.xlabel("Predicted label")
        plt.ylabel("True label")
        plt.colorbar()

        if class_names and len(class_names) == cm.shape[0]:
            ticks = np.arange(len(class_names))
            plt.xticks(ticks, class_names, rotation=45, ha="right")
            plt.yticks(ticks, class_names)

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, str(cm[i, j]), ha="center", va="center")

        plt.tight_layout()

        fig_path = confusion_output_dir / f"{experiment_id}_{split}_confusion_matrix.png"
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.show()

        saved_confusion_figures.append(fig_path)

print(f"Saved {len(saved_confusion_figures)} confusion matrix figures")

## Final files produced

In [ ]:
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(p)